In [1]:
from __future__ import annotations

from pathlib import Path
from collections import defaultdict
import pandas as pd
import re
import json
import hashlib
from dbfread import DBF


CSV_ENCODINGS = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
DBF_ENCODINGS = ["latin-1", "cp1252", "iso-8859-1", "utf-8"]

In [2]:
ROOT = Path("../data/raw/ENSU")

ensu_cs_files = sorted([
    f for f in ROOT.rglob("*")
    if f.is_file() and "ENSU_CS" in f.name.upper() and f.suffix.lower() in [".csv", ".dbf"]
])

print("Total archivos ENSU_CS:", len(ensu_cs_files))
for f in ensu_cs_files:
    print(f.name)

Total archivos ENSU_CS: 48
ENSU_CS0316.dbf
ENSU_CS0616.dbf
ENSU_CS_0314.DBF
ENSU_CS_0315.DBF
ENSU_CS_0317.dbf
ENSU_CS_0318.dbf
ENSU_CS_0319.dbf
ENSU_CS_0320.dbf
ENSU_CS_0322.csv
ENSU_CS_0323.csv
ENSU_CS_0324.csv
ENSU_CS_0325.csv
ENSU_CS_0614.DBF
ENSU_CS_0615.DBF
ENSU_CS_0617.dbf
ENSU_CS_0618.dbf
ENSU_CS_0619.dbf
ENSU_CS_0621.csv
ENSU_CS_0622.csv
ENSU_CS_0623.csv
ENSU_CS_0624.csv
ENSU_CS_0625.csv
ENSU_CS_0913.DBF
ENSU_CS_0914.DBF
ENSU_CS_0915.DBF
ENSU_CS_0916.dbf
ENSU_CS_0917.dbf
ENSU_CS_0918.dbf
ENSU_CS_0919.dbf
ENSU_CS_0920.dbf
ENSU_CS_0921.csv
ENSU_CS_0922.csv
ENSU_CS_0923.csv
ENSU_CS_0924.csv
ENSU_CS_0925.csv
ENSU_CS_1213.DBF
ENSU_CS_1214.DBF
ENSU_CS_1215.DBF
ENSU_CS_1216.dbf
ENSU_CS_1217.dbf
ENSU_CS_1218.dbf
ENSU_CS_1219.dbf
ENSU_CS_1220.dbf
ENSU_CS_1221.csv
ENSU_CS_1222.csv
ENSU_CS_1223.csv
ENSU_CS_1224.csv
ENSU_CS_1225.csv


In [3]:
def read_csv_robust(path):
    last_error = None
    for enc in CSV_ENCODINGS:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            return df, f"csv:{enc}"
        except Exception as e:
            last_error = e
    raise last_error

def read_dbf_robust(path):
    last_error = None
    
    for enc in DBF_ENCODINGS:
        try:
            table = DBF(str(path), load=True, encoding=enc, char_decode_errors="strict")
            df = pd.DataFrame(iter(table))
            return df, f"dbf:{enc}"
        except Exception as e:
            last_error = e

    for enc in DBF_ENCODINGS:
        try:
            table = DBF(str(path), load=True, encoding=enc, char_decode_errors="ignore")
            df = pd.DataFrame(iter(table))
            return df, f"dbf:{enc}:ignore"
        except Exception as e:
            last_error = e

    raise last_error

def read_file(path):
    if path.suffix.lower() == ".csv":
        return read_csv_robust(path)
    elif path.suffix.lower() == ".dbf":
        return read_dbf_robust(path)
    else:
        raise ValueError(f"Extensión no soportada: {path.suffix}")

def normalize_columns(columns):
    return (
        pd.Index(columns)
        .map(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", "_", regex=True)
        .tolist()
    )

def parse_period_from_filename(filename):
    name = Path(filename).stem.upper()

    patterns = [
        r"_(\d{2})(\d{2})$",         # ENSU_CS_0314
        r"CS(\d{2})(\d{2})$",        # ENSU_CS0316
        r"_(\d{2})(\d{2})",          # fallback
        r"CS(\d{2})(\d{2})",         # fallback
    ]

    for pat in patterns:
        m = re.search(pat, name)
        if m:
            month = int(m.group(1))
            yy = int(m.group(2))
            year = 2000 + yy if yy <= 79 else 1900 + yy
            quarter = ((month - 1) // 3 + 1) if 1 <= month <= 12 else None
            return year, month, quarter

    return None, None, None

In [4]:
records = []

for f in ensu_cs_files:
    print("Leyendo:", f.name)
    
    df, source_info = read_file(f)
    year, month, quarter = parse_period_from_filename(f.name)
    cols = normalize_columns(df.columns)
    
    records.append({
        "file": f.name,
        "year": year,
        "month": month,
        "quarter": quarter,
        "source_info": source_info,
        "n_rows": len(df),
        "n_cols": len(cols),
        "columns": cols,
        "columns_set": frozenset(cols),
    })

audit_df = pd.DataFrame(records)

audit_df[["file", "year", "month", "quarter", "source_info", "n_rows", "n_cols"]]

Leyendo: ENSU_CS0316.dbf
Leyendo: ENSU_CS0616.dbf
Leyendo: ENSU_CS_0314.DBF
Leyendo: ENSU_CS_0315.DBF
Leyendo: ENSU_CS_0317.dbf
Leyendo: ENSU_CS_0318.dbf
Leyendo: ENSU_CS_0319.dbf
Leyendo: ENSU_CS_0320.dbf
Leyendo: ENSU_CS_0322.csv
Leyendo: ENSU_CS_0323.csv
Leyendo: ENSU_CS_0324.csv
Leyendo: ENSU_CS_0325.csv
Leyendo: ENSU_CS_0614.DBF
Leyendo: ENSU_CS_0615.DBF
Leyendo: ENSU_CS_0617.dbf
Leyendo: ENSU_CS_0618.dbf
Leyendo: ENSU_CS_0619.dbf
Leyendo: ENSU_CS_0621.csv
Leyendo: ENSU_CS_0622.csv
Leyendo: ENSU_CS_0623.csv
Leyendo: ENSU_CS_0624.csv
Leyendo: ENSU_CS_0625.csv
Leyendo: ENSU_CS_0913.DBF
Leyendo: ENSU_CS_0914.DBF
Leyendo: ENSU_CS_0915.DBF
Leyendo: ENSU_CS_0916.dbf
Leyendo: ENSU_CS_0917.dbf
Leyendo: ENSU_CS_0918.dbf
Leyendo: ENSU_CS_0919.dbf
Leyendo: ENSU_CS_0920.dbf
Leyendo: ENSU_CS_0921.csv
Leyendo: ENSU_CS_0922.csv
Leyendo: ENSU_CS_0923.csv
Leyendo: ENSU_CS_0924.csv
Leyendo: ENSU_CS_0925.csv
Leyendo: ENSU_CS_1213.DBF
Leyendo: ENSU_CS_1214.DBF
Leyendo: ENSU_CS_1215.DBF
Leyendo: ENSU_

,file,year,month,quarter,source_info,n_rows,n_cols
0,ENSU_CS0316.dbf,2016,3,1,dbf:latin-1,34200,44
1,ENSU_CS0616.dbf,2016,6,2,dbf:latin-1,43005,38
2,ENSU_CS_0314.DBF,2014,3,1,dbf:latin-1,7255,32
3,ENSU_CS_0315.DBF,2015,3,1,dbf:latin-1,7336,32
4,ENSU_CS_0317.dbf,2017,3,1,dbf:latin-1,49675,40
5,ENSU_CS_0318.dbf,2018,3,1,dbf:latin-1,51765,40
6,ENSU_CS_0319.dbf,2019,3,1,dbf:latin-1,61036,40
7,ENSU_CS_0320.dbf,2020,3,1,dbf:latin-1,74054,37
8,ENSU_CS_0322.csv,2022,3,1,csv:utf-8,75835,41
9,ENSU_CS_0323.csv,2023,3,1,csv:utf-8,75759,42


In [5]:
audit_df.sort_values(["year", "month"])[["file", "year", "month", "quarter", "n_cols"]]

,file,year,month,quarter,n_cols
22,ENSU_CS_0913.DBF,2013,9,3,32
35,ENSU_CS_1213.DBF,2013,12,4,32
2,ENSU_CS_0314.DBF,2014,3,1,32
12,ENSU_CS_0614.DBF,2014,6,2,32
23,ENSU_CS_0914.DBF,2014,9,3,32
36,ENSU_CS_1214.DBF,2014,12,4,32
3,ENSU_CS_0315.DBF,2015,3,1,32
13,ENSU_CS_0615.DBF,2015,6,2,32
24,ENSU_CS_0915.DBF,2015,9,3,32
37,ENSU_CS_1215.DBF,2015,12,4,32


In [6]:
year_summary = []

for year, g in audit_df.groupby("year"):
    unique_structures = g["columns_set"].nunique()
    
    year_summary.append({
        "year": year,
        "n_files": len(g),
        "months": sorted(g["month"].tolist()),
        "quarters": sorted(g["quarter"].tolist()),
        "n_distinct_structures": unique_structures,
        "all_equal_within_year": unique_structures == 1
    })

year_summary_df = pd.DataFrame(year_summary).sort_values("year")
year_summary_df

,year,n_files,months,quarters,n_distinct_structures,all_equal_within_year
0,2013,2,"[9, 12]","[3, 4]",1,True
1,2014,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
2,2015,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",2,False
3,2016,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",3,False
4,2017,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",2,False
5,2018,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
6,2019,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
7,2020,3,"[3, 9, 12]","[1, 3, 4]",2,False
8,2021,3,"[6, 9, 12]","[2, 3, 4]",1,True
9,2022,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",2,False


In [7]:
def analyze_year_structure(audit_df, year):
    tmp = audit_df[audit_df["year"] == year].copy().sort_values("month")

    column_sets = [set(x) for x in tmp["columns_set"].tolist()]

    common_cols = set.intersection(*column_sets)
    all_cols = set.union(*column_sets)
    inconsistent_cols = all_cols - common_cols

    result = {
        "year": year,
        "n_files": len(tmp),
        "files": tmp["file"].tolist(),
        "months": tmp["month"].tolist(),
        "n_common_cols": len(common_cols),
        "n_inconsistent_cols": len(inconsistent_cols),
        "common_cols": sorted(common_cols),
        "inconsistent_cols": sorted(inconsistent_cols),
    }
    return result


def column_presence_by_file(audit_df, year):
    tmp = audit_df[audit_df["year"] == year].copy().sort_values("month")

    column_sets = [set(x) for x in tmp["columns_set"].tolist()]
    columns = sorted(set.union(*column_sets))

    matrix = []
    for col in columns:
        row = {"column": col}
        for _, r in tmp.iterrows():
            row[r["file"]] = col in set(r["columns_set"])
        matrix.append(row)

    return pd.DataFrame(matrix)

In [8]:
problem_years = year_summary_df.loc[
    ~year_summary_df["all_equal_within_year"], "year"
].tolist()

year_diffs = []
presence_tables = {}

for y in problem_years:
    res = analyze_year_structure(audit_df, y)
    year_diffs.append({
        "year": res["year"],
        "n_files": res["n_files"],
        "months": res["months"],
        "n_common_cols": res["n_common_cols"],
        "n_inconsistent_cols": res["n_inconsistent_cols"],
        "inconsistent_cols": res["inconsistent_cols"],
        "files": res["files"],
    })
    presence_tables[y] = column_presence_by_file(audit_df, y)

year_diffs_df = pd.DataFrame(year_diffs).sort_values("year")
year_diffs_df


,year,n_files,months,n_common_cols,n_inconsistent_cols,inconsistent_cols,files
0,2015,4,"[3, 6, 9, 12]",31,2,"[FAC_MOD, FAC_P18]","[ENSU_CS_0315.DBF, ENSU_CS_0615.DBF, ENSU_CS_0..."
1,2016,4,"[3, 6, 9, 12]",32,20,"[AGEB, AP3_4A, COD_SEL, CVE_ENT, CVE_MUN, ENT,...","[ENSU_CS0316.dbf, ENSU_CS0616.dbf, ENSU_CS_091..."
2,2017,4,"[3, 6, 9, 12]",39,2,"[EDA, EDAD]","[ENSU_CS_0317.dbf, ENSU_CS_0617.dbf, ENSU_CS_0..."
3,2020,3,"[3, 9, 12]",36,5,"[ID_PER, ID_VIV, NOM_CD, R_INF, VERIF_COD]","[ENSU_CS_0320.dbf, ENSU_CS_0920.dbf, ENSU_CS_1..."
4,2022,4,"[3, 6, 9, 12]",41,1,[REG_CD],"[ENSU_CS_0322.csv, ENSU_CS_0622.csv, ENSU_CS_0..."
5,2023,4,"[3, 6, 9, 12]",41,1,[UNNAMED:_0],"[ENSU_CS_0323.csv, ENSU_CS_0623.csv, ENSU_CS_0..."
6,2024,4,"[3, 6, 9, 12]",40,3,"[ID_PERT, ID_VIVT, REG_CD]","[ENSU_CS_0324.csv, ENSU_CS_0624.csv, ENSU_CS_0..."
7,2025,4,"[3, 6, 9, 12]",39,5,"[CVEGEO, CVE_LOC, ID_PERT, ID_VIVT, LOC]","[ENSU_CS_0325.csv, ENSU_CS_0625.csv, ENSU_CS_0..."


In [9]:
for _, row in year_diffs_df.iterrows():
    print("=" * 80)
    print(f"AÑO {row['year']}")
    print(f"n_files: {row['n_files']}")
    print(f"n_common_cols: {row['n_common_cols']}")
    print(f"n_inconsistent_cols: {row['n_inconsistent_cols']}")
    print("Columnas conflictivas:")
    print(row["inconsistent_cols"])

AÑO 2015
n_files: 4
n_common_cols: 31
n_inconsistent_cols: 2
Columnas conflictivas:
['FAC_MOD', 'FAC_P18']
AÑO 2016
n_files: 4
n_common_cols: 32
n_inconsistent_cols: 20
Columnas conflictivas:
['AGEB', 'AP3_4A', 'COD_SEL', 'CVE_ENT', 'CVE_MUN', 'ENT', 'ERR_SEL', 'FCHDEFDIA', 'FCHDEFH_I', 'FCHDEFH_T', 'FCHDEFMES', 'FCHDEFM_I', 'FCHDEFM_T', 'MUN', 'NAC_A', 'NOM_ENT', 'NOM_MUN', 'PRO_VIV', 'SEL_SUST', 'VERIF_COD']
AÑO 2017
n_files: 4
n_common_cols: 39
n_inconsistent_cols: 2
Columnas conflictivas:
['EDA', 'EDAD']
AÑO 2020
n_files: 3
n_common_cols: 36
n_inconsistent_cols: 5
Columnas conflictivas:
['ID_PER', 'ID_VIV', 'NOM_CD', 'R_INF', 'VERIF_COD']
AÑO 2022
n_files: 4
n_common_cols: 41
n_inconsistent_cols: 1
Columnas conflictivas:
['REG_CD']
AÑO 2023
n_files: 4
n_common_cols: 41
n_inconsistent_cols: 1
Columnas conflictivas:
['UNNAMED:_0']
AÑO 2024
n_files: 4
n_common_cols: 40
n_inconsistent_cols: 3
Columnas conflictivas:
['ID_PERT', 'ID_VIVT', 'REG_CD']
AÑO 2025
n_files: 4
n_common_cols: 39


In [10]:
RENAME_MAP = {
    # equivalencias claras
    "EDA": "EDAD",
    "ID_PERT": "ID_PER",
    "ID_VIVT": "ID_VIV",
    "ENT": "CVE_ENT",
    "MUN": "CVE_MUN",
    "CVE_LOC": "LOC",   # o al revés, pero hay que unificar a uno
}

DROP_COLS_GLOBAL = {
    # columnas claramente auxiliares / ruido
    "UNNAMED:_0",
    "REG_CD",
    "FAC_MOD",
    "FAC_P18",
    "R_INF",
    "VERIF_COD",
    "ERR_SEL",
    "SEL_SUST",
    "FCHDEFDIA",
    "FCHDEFH_I",
    "FCHDEFH_T",
    "FCHDEFMES",
    "FCHDEFM_I",
    "FCHDEFM_T",
}

# columnas que por ahora NO vamos a tocar automáticamente
# porque podrían ser útiles y hay que revisar antes
REVIEW_COLS = {
    "AGEB",
    "AP3_4A",
    "PRO_VIV",
    "NAC_A",
    "CVEGEO",
    "NOM_CD",
}

In [11]:
def apply_harmonization_rules(cols):
    cols = [RENAME_MAP.get(c, c) for c in cols]
    cols = [c for c in cols if c not in DROP_COLS_GLOBAL]
    return cols


harm_records = []

for _, row in audit_df.iterrows():
    cols = list(row["columns"])
    cols = apply_harmonization_rules(cols)

    harm_records.append({
        "file": row["file"],
        "year": row["year"],
        "month": row["month"],
        "quarter": row["quarter"],
        "n_cols_harmonized": len(cols),
        "columns_harmonized": cols,
        "columns_set_harmonized": frozenset(cols),
    })

audit_harmonized_df = pd.DataFrame(harm_records)
audit_harmonized_df.head()

,file,year,month,quarter,n_cols_harmonized,columns_harmonized,columns_set_harmonized
0,ENSU_CS0316.dbf,2016,3,1,36,"[VIV_SEL, H_MUD, CD, CVE_ENT, CVE_MUN, LOC, AG...","frozenset({EST_DIS, UPM_DIS, C_RES, N_REN, VIV..."
1,ENSU_CS0616.dbf,2016,6,2,35,"[UPM, VIV_SEL, CVE_ENT, CVE_MUN, LOC, CD, PER,...","frozenset({EST_DIS, UPM_DIS, C_RES, N_REN, VIV..."
2,ENSU_CS_0314.DBF,2014,3,1,31,"[PER, N_ENT, FOL, CVE_ENT, CON, V_SEL, N_HOG, ...","frozenset({UPM_DIS, EDIS, C_RES, A_ECO, N_REN,..."
3,ENSU_CS_0315.DBF,2015,3,1,31,"[PER, N_ENT, FOL, CVE_ENT, CON, V_SEL, N_HOG, ...","frozenset({UPM_DIS, EDIS, C_RES, A_ECO, N_REN,..."
4,ENSU_CS_0317.dbf,2017,3,1,37,"[UPM, VIV_SEL, CVE_ENT, NOM_ENT, CVE_MUN, NOM_...","frozenset({EST_DIS, UPM_DIS, C_RES, N_REN, VIV..."


In [12]:
year_summary_harmonized = []

for year, g in audit_harmonized_df.groupby("year"):
    unique_structures = g["columns_set_harmonized"].nunique()

    year_summary_harmonized.append({
        "year": year,
        "n_files": len(g),
        "months": sorted(g["month"].tolist()),
        "quarters": sorted(g["quarter"].tolist()),
        "n_distinct_structures": unique_structures,
        "all_equal_within_year": unique_structures == 1
    })

year_summary_harmonized_df = pd.DataFrame(year_summary_harmonized).sort_values("year")
year_summary_harmonized_df

,year,n_files,months,quarters,n_distinct_structures,all_equal_within_year
0,2013,2,"[9, 12]","[3, 4]",1,True
1,2014,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
2,2015,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
3,2016,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",3,False
4,2017,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
5,2018,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
6,2019,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
7,2020,3,"[3, 9, 12]","[1, 3, 4]",2,False
8,2021,3,"[6, 9, 12]","[2, 3, 4]",1,True
9,2022,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True


In [13]:
problem_years_harmonized = year_summary_harmonized_df.loc[
    ~year_summary_harmonized_df["all_equal_within_year"], "year"
].tolist()

problem_years_harmonized

[2016, 2020, 2025]

In [14]:
def analyze_year_structure_harmonized(audit_harmonized_df, year):
    tmp = audit_harmonized_df[audit_harmonized_df["year"] == year].copy().sort_values("month")

    column_sets = [set(x) for x in tmp["columns_set_harmonized"].tolist()]

    common_cols = set.intersection(*column_sets)
    all_cols = set.union(*column_sets)
    inconsistent_cols = sorted(all_cols - common_cols)

    return {
        "year": year,
        "files": tmp["file"].tolist(),
        "n_files": len(tmp),
        "n_common_cols": len(common_cols),
        "n_inconsistent_cols": len(inconsistent_cols),
        "inconsistent_cols": inconsistent_cols,
    }


for y in problem_years_harmonized:
    res = analyze_year_structure_harmonized(audit_harmonized_df, y)
    print("=" * 80)
    print(f"AÑO {y}")
    print(f"n_common_cols: {res['n_common_cols']}")
    print(f"n_inconsistent_cols: {res['n_inconsistent_cols']}")
    print(res["inconsistent_cols"])

AÑO 2016
n_common_cols: 33
n_inconsistent_cols: 7
['AGEB', 'AP3_4A', 'COD_SEL', 'NAC_A', 'NOM_ENT', 'NOM_MUN', 'PRO_VIV']
AÑO 2020
n_common_cols: 36
n_inconsistent_cols: 3
['ID_PER', 'ID_VIV', 'NOM_CD']
AÑO 2025
n_common_cols: 39
n_inconsistent_cols: 1
['CVEGEO']


In [15]:
DROP_COLS_GLOBAL.add("CVEGEO")

RENAME_MAP.update({
    "ID_PER": "PER",
    "ID_VIV": "VIV_SEL"
})

DROP_COLS_GLOBAL.add("NOM_CD")

DROP_COLS_GLOBAL.update({
    "AGEB",
    "PRO_VIV",
    "COD_SEL",
    "NOM_ENT",
    "NOM_MUN",
    "AP3_4A",
    "NAC_A"
})

In [16]:
harm_records = []

for _, row in audit_df.iterrows():
    cols = list(row["columns"])
    cols = apply_harmonization_rules(cols)

    harm_records.append({
        "file": row["file"],
        "year": row["year"],
        "month": row["month"],
        "quarter": row["quarter"],
        "n_cols_harmonized": len(cols),
        "columns_harmonized": cols,
        "columns_set_harmonized": frozenset(cols),
    })

audit_harmonized_df = pd.DataFrame(harm_records)

In [17]:
year_summary_harmonized = []

for year, g in audit_harmonized_df.groupby("year"):
    unique_structures = g["columns_set_harmonized"].nunique()

    year_summary_harmonized.append({
        "year": year,
        "n_files": len(g),
        "months": sorted(g["month"].tolist()),
        "quarters": sorted(g["quarter"].tolist()),
        "n_distinct_structures": unique_structures,
        "all_equal_within_year": unique_structures == 1
    })

year_summary_harmonized_df = (
    pd.DataFrame(year_summary_harmonized)
    .sort_values("year")
    .reset_index(drop=True)
)

year_summary_harmonized_df

,year,n_files,months,quarters,n_distinct_structures,all_equal_within_year
0,2013,2,"[9, 12]","[3, 4]",1,True
1,2014,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
2,2015,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
3,2016,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
4,2017,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
5,2018,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
6,2019,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
7,2020,3,"[3, 9, 12]","[1, 3, 4]",1,True
8,2021,3,"[6, 9, 12]","[2, 3, 4]",1,True
9,2022,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True


In [18]:
problem_years_harmonized = year_summary_harmonized_df.loc[
    ~year_summary_harmonized_df["all_equal_within_year"], "year"
].tolist()

problem_years_harmonized

[2024, 2025]

In [19]:
for y in problem_years_harmonized:
    res = analyze_year_structure_harmonized(audit_harmonized_df, y)
    print("=" * 80)
    print(f"AÑO {y}")
    print(f"n_common_cols: {res['n_common_cols']}")
    print(f"n_inconsistent_cols: {res['n_inconsistent_cols']}")
    print(res["inconsistent_cols"])

AÑO 2024
n_common_cols: 33
n_inconsistent_cols: 2
['ID_PER', 'ID_VIV']
AÑO 2025
n_common_cols: 33
n_inconsistent_cols: 2
['ID_PER', 'ID_VIV']


In [20]:
RENAME_MAP = {
    "EDA": "EDAD",
    "ENT": "CVE_ENT",
    "MUN": "CVE_MUN",
    "CVE_LOC": "LOC",

    "ID_PERT": "PER",
    "ID_PER": "PER",

    "ID_VIVT": "VIV_SEL",
    "ID_VIV": "VIV_SEL",
}

DROP_COLS_GLOBAL = {
    "UNNAMED:_0",
    "REG_CD",
    "FAC_MOD",
    "FAC_P18",
    "R_INF",
    "VERIF_COD",
    "ERR_SEL",
    "SEL_SUST",
    "FCHDEFDIA",
    "FCHDEFH_I",
    "FCHDEFH_T",
    "FCHDEFMES",
    "FCHDEFM_I",
    "FCHDEFM_T",
    "AGEB",
    "PRO_VIV",
    "COD_SEL",
    "NOM_ENT",
    "NOM_MUN",
    "AP3_4A",
    "NAC_A",
    "NOM_CD",
    "CVEGEO",
}

In [21]:
def apply_harmonization_rules(cols):
    cols = [RENAME_MAP.get(c, c) for c in cols]
    cols = [c for c in cols if c not in DROP_COLS_GLOBAL]
    return cols

harm_records = []

for _, row in audit_df.iterrows():
    cols = list(row["columns"])
    cols = apply_harmonization_rules(cols)

    harm_records.append({
        "file": row["file"],
        "year": row["year"],
        "month": row["month"],
        "quarter": row["quarter"],
        "n_cols_harmonized": len(cols),
        "columns_harmonized": cols,
        "columns_set_harmonized": frozenset(cols),
    })

audit_harmonized_df = pd.DataFrame(harm_records)

year_summary_harmonized = []

for year, g in audit_harmonized_df.groupby("year"):
    unique_structures = g["columns_set_harmonized"].nunique()

    year_summary_harmonized.append({
        "year": year,
        "n_files": len(g),
        "months": sorted(g["month"].tolist()),
        "quarters": sorted(g["quarter"].tolist()),
        "n_distinct_structures": unique_structures,
        "all_equal_within_year": unique_structures == 1
    })

year_summary_harmonized_df = (
    pd.DataFrame(year_summary_harmonized)
    .sort_values("year")
    .reset_index(drop=True)
)

year_summary_harmonized_df

,year,n_files,months,quarters,n_distinct_structures,all_equal_within_year
0,2013,2,"[9, 12]","[3, 4]",1,True
1,2014,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
2,2015,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
3,2016,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
4,2017,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
5,2018,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
6,2019,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True
7,2020,3,"[3, 9, 12]","[1, 3, 4]",1,True
8,2021,3,"[6, 9, 12]","[2, 3, 4]",1,True
9,2022,4,"[3, 6, 9, 12]","[1, 2, 3, 4]",1,True


In [22]:
def collapse_duplicate_columns(df):
    """
    Si después de renombrar quedan columnas duplicadas,
    las combina tomando el primer valor no nulo por fila.
    """
    cols = pd.Series(df.columns)
    duplicated_names = cols[cols.duplicated()].unique().tolist()

    if not duplicated_names:
        return df

    df = df.copy()

    for col in duplicated_names:
        same_name_cols = df.loc[:, df.columns == col]

        # combinar por fila tomando el primer no nulo
        merged = same_name_cols.bfill(axis=1).iloc[:, 0]

        # eliminar todas las columnas con ese nombre
        df = df.loc[:, df.columns != col]

        # agregar una sola columna consolidada
        df[col] = merged

    return df

In [23]:
yearly_data = {}

for year, g in audit_df.groupby("year"):
    g = g.sort_values("month").copy()
    dfs_year = []

    for _, r in g.iterrows():
        file_matches = [p for p in ensu_cs_files if p.name == r["file"]]
        if not file_matches:
            raise FileNotFoundError(f"No se encontró el archivo {r['file']}")
        
        path = file_matches[0]

        df, source_info = read_file(path)
        df.columns = normalize_columns(df.columns)

        # aplicar armonización final
        df.columns = [RENAME_MAP.get(c, c) for c in df.columns]

        # colapsar duplicados generados por renombrado
        df = collapse_duplicate_columns(df)

        # eliminar columnas globales
        df = df.drop(columns=[c for c in DROP_COLS_GLOBAL if c in df.columns], errors="ignore")

        # metadatos del trimestre
        df["YEAR"] = r["year"]
        df["MONTH"] = r["month"]
        df["QUARTER"] = r["quarter"]
        df["SOURCE_FILE"] = r["file"]

        dfs_year.append(df)

    yearly_df = pd.concat(dfs_year, ignore_index=True, sort=False)
    yearly_data[year] = yearly_df

In [24]:
yearly_shapes = pd.DataFrame([
    {
        "year": year,
        "rows": df.shape[0],
        "cols": df.shape[1]
    }
    for year, df in yearly_data.items()
]).sort_values("year")

yearly_shapes

,year,rows,cols
0,2013,14543,35
1,2014,29454,35
2,2015,29863,35
3,2016,170399,37
4,2017,205363,37
5,2018,240489,37
6,2019,273942,37
7,2020,219191,37
8,2021,223353,37
9,2022,304610,37


In [25]:
# Crear la carpeta de salida si no existe
output_dir = Path("../data/raw/ENSU/")
output_dir.mkdir(parents=True, exist_ok=True)

# Exportar cada DataFrame anual
for year, df in yearly_data.items():
    # Nombre del archivo: ENSU_annual_YYYY.csv
    output_file = output_dir / f"ENSU_annual_{year}.csv"
    
    # Exportar a CSV sin índice
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    print(f"Exportado: {output_file.name} - {len(df)} filas, {len(df.columns)} columnas")

# Verificar los archivos exportados
print("\nArchivos exportados:")
exported_files = sorted(output_dir.glob("ENSU_annual_*.csv"))
for f in exported_files:
    print(f"  - {f.name}")

Exportado: ENSU_annual_2013.csv - 14543 filas, 35 columnas
Exportado: ENSU_annual_2014.csv - 29454 filas, 35 columnas
Exportado: ENSU_annual_2015.csv - 29863 filas, 35 columnas
Exportado: ENSU_annual_2016.csv - 170399 filas, 37 columnas
Exportado: ENSU_annual_2017.csv - 205363 filas, 37 columnas
Exportado: ENSU_annual_2018.csv - 240489 filas, 37 columnas
Exportado: ENSU_annual_2019.csv - 273942 filas, 37 columnas
Exportado: ENSU_annual_2020.csv - 219191 filas, 37 columnas
Exportado: ENSU_annual_2021.csv - 223353 filas, 37 columnas
Exportado: ENSU_annual_2022.csv - 304610 filas, 37 columnas
Exportado: ENSU_annual_2023.csv - 306584 filas, 37 columnas
Exportado: ENSU_annual_2024.csv - 301124 filas, 37 columnas
Exportado: ENSU_annual_2025.csv - 296821 filas, 37 columnas

Archivos exportados:
  - ENSU_annual_2013.csv
  - ENSU_annual_2014.csv
  - ENSU_annual_2015.csv
  - ENSU_annual_2016.csv
  - ENSU_annual_2017.csv
  - ENSU_annual_2018.csv
  - ENSU_annual_2019.csv
  - ENSU_annual_2020.csv
 